# **Modeling — Multi-Modal Fusion Stacking Ensemble**
**Tahap:** Load hasil feature engineering → Tier 1 (XGBoost + LightGBM) → Tier 2 (Meta-Learner)

**Prasyarat:** Jalankan `FEATURE_ENGINEERING.ipynb` terlebih dahulu.

Output file yang dihasilkan:
- `meta_learner.pkl` — model Meta-Learner
- `model_fisik.pkl` — model XGBoost
- `model_kimia.pkl` — model LightGBM
- `hasil_prediksi.csv` — hasil prediksi test set

## **Import Library**

In [9]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

## **Load Hasil Feature Engineering**

In [2]:
X_train_fisik     = pd.read_csv('../prepare/X_train_fisik.csv')
X_test_fisik      = pd.read_csv('../prepare/X_test_fisik.csv')

X_train_kimia     = pd.read_csv('../prepare/X_train_kimia.csv')
X_test_kimia      = pd.read_csv('../prepare/X_test_kimia.csv')

X_train_demografi = pd.read_csv('../prepare/X_train_demografi.csv')
X_test_demografi  = pd.read_csv('../prepare/X_test_demografi.csv')

y_train = np.load('../prepare/y_train.npy')
y_test  = np.load('../prepare/y_test.npy')

print('Data berhasil dimuat!')
print(f'  Fisik    — train: {X_train_fisik.shape}, test: {X_test_fisik.shape}')
print(f'  Kimia    — train: {X_train_kimia.shape}, test: {X_test_kimia.shape}')
print(f'  Demografi— train: {X_train_demografi.shape}, test: {X_test_demografi.shape}')

Data berhasil dimuat!
  Fisik    — train: (255, 9), test: (64, 9)
  Kimia    — train: (255, 13), test: (64, 13)
  Demografi— train: (255, 7), test: (64, 7)


# **Tier 1**

## **Sub-Model A: XGBoost (Modalitas Fisik)**
Probabilitas `P_fisik` dihasilkan lewat **5-Fold Cross Validation** agar tidak terjadi data leakage ke Tier 2.

In [3]:
model_fisik = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Meta-fitur P_fisik untuk train (via cross_val_predict)
P_fisik_train = cross_val_predict(
    model_fisik, X_train_fisik, y_train,
    cv=cv, method='predict_proba'
)[:, 1]

# Fit ulang pada seluruh train → untuk prediksi test
model_fisik.fit(X_train_fisik, y_train)
P_fisik_test = model_fisik.predict_proba(X_test_fisik)[:, 1]

print(f'P_fisik_train — mean: {P_fisik_train.mean():.4f}, std: {P_fisik_train.std():.4f}')
print(f'P_fisik_test  — mean: {P_fisik_test.mean():.4f}, std: {P_fisik_test.std():.4f}')

P_fisik_train — mean: 0.4797, std: 0.3225
P_fisik_test  — mean: 0.5259, std: 0.2966


## **Sub-Model B: LightGBM (Modalitas Kimia)**

In [4]:
model_kimia = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

# Meta-fitur P_kimia untuk train (via cross_val_predict)
P_kimia_train = cross_val_predict(
    model_kimia, X_train_kimia, y_train,
    cv=cv, method='predict_proba'
)[:, 1]

# Fit ulang pada seluruh train → untuk prediksi test
model_kimia.fit(X_train_kimia, y_train)
P_kimia_test = model_kimia.predict_proba(X_test_kimia)[:, 1]

print(f'P_kimia_train — mean: {P_kimia_train.mean():.4f}, std: {P_kimia_train.std():.4f}')
print(f'P_kimia_test  — mean: {P_kimia_test.mean():.4f}, std: {P_kimia_test.std():.4f}')

P_kimia_train — mean: 0.4924, std: 0.3619
P_kimia_test  — mean: 0.4626, std: 0.3657


## **Pembentukan Matriks Tier 2**
Gabungkan `P_fisik + P_kimia + Modalitas C (Demografis)` → **9 kolom input Meta-Learner**.

In [5]:
# Train
meta_train = pd.DataFrame({'P_fisik': P_fisik_train, 'P_kimia': P_kimia_train})
X_tier2_train = pd.concat([meta_train, X_train_demografi.reset_index(drop=True)], axis=1)

# Test
meta_test = pd.DataFrame({'P_fisik': P_fisik_test, 'P_kimia': P_kimia_test})
X_tier2_test = pd.concat([meta_test, X_test_demografi.reset_index(drop=True)], axis=1)

print('Matriks Tier 2 berhasil dibentuk!')
print(f'  Shape Train : {X_tier2_train.shape}')
print(f'  Shape Test  : {X_tier2_test.shape}')
print(f'  Kolom       : {list(X_tier2_train.columns)}')
print()
X_tier2_train.head()

Matriks Tier 2 berhasil dibentuk!
  Shape Train : (255, 9)
  Shape Test  : (64, 9)
  Kolom       : ['P_fisik', 'P_kimia', 'Age', 'Gender', 'Comorbidity', 'Coronary Artery Disease (CAD)', 'Hypothyroidism', 'Hyperlipidemia', 'Diabetes Mellitus (DM)']



,P_fisik,P_kimia,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM)
0,0.983871,0.993319,0.957108,1.003929,-0.668013,-0.179969,-0.191273,-0.15523,-0.398862
1,0.713556,0.310020,-1.146880,-0.996086,1.312723,-0.179969,-0.191273,-0.15523,-0.398862
2,0.581666,0.313208,0.228805,-0.996086,1.312723,-0.179969,-0.191273,-0.15523,2.507133
3,0.163191,0.432994,0.228805,-0.996086,-0.668013,-0.179969,-0.191273,-0.15523,-0.398862
4,0.882492,0.989263,0.390650,1.003929,1.312723,-0.179969,-0.191273,-0.15523,2.507133


## **Meta-Learner: Logistic Regression**
Meta-Learner mendamaikan keputusan Sub-Model A dan B, diperkuat konteks demografis pasien.

In [6]:
meta_learner = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_learner.fit(X_tier2_train, y_train)

y_pred       = meta_learner.predict(X_tier2_test)
y_pred_proba = meta_learner.predict_proba(X_tier2_test)[:, 1]

print('Meta-Learner selesai dilatih!')

Meta-Learner selesai dilatih!


## **Simpan Output untuk EVALUASI.ipynb**

In [8]:
# Simpan model
joblib.dump(model_fisik,   '../save_model/model_fisik.pkl')
joblib.dump(model_kimia,   '../save_model/model_kimia.pkl')
joblib.dump(meta_learner,  '../save_model/meta_learner.pkl')

# Simpan hasil prediksi & meta-fitur
hasil = pd.DataFrame({
    'y_test'      : y_test,
    'y_pred'      : y_pred,
    'y_pred_proba': y_pred_proba,
    'P_fisik_test': P_fisik_test,
    'P_kimia_test': P_kimia_test
})
hasil.to_csv('../result/hasil_prediksi.csv', index=False)

# Simpan koefisien meta-learner
coef_df = pd.DataFrame({
    'Fitur'     : X_tier2_train.columns,
    'Koefisien' : meta_learner.coef_[0]
})
coef_df.to_csv('../result/meta_learner_coef.csv', index=False)

print('Semua file berhasil disimpan!')
print('  model_fisik.pkl, model_kimia.pkl, meta_learner.pkl')
print('  hasil_prediksi.csv')
print('  meta_learner_coef.csv')

Semua file berhasil disimpan!
  model_fisik.pkl, model_kimia.pkl, meta_learner.pkl
  hasil_prediksi.csv
  meta_learner_coef.csv
